# 08 — Stacks, Queues, and Deques

## Why This Matters for DSA
This notebook closes Phase 02 by formalizing three structures you've already been using WITHOUT naming them. The call stack that powered every recursive function in `02_Recursion` (Phase 01) IS a stack — LIFO access is what makes recursion work. The queue that sits behind every BFS traversal you'll meet in Phase 03/04 IS a queue — FIFO access is what guarantees level-order processing. And `03_STL_Deep_Dive` (Phase 00, Section 7) already showed you the C++ adapters (`std::stack`, `std::queue`) that wrap these access patterns — this notebook is where you learn WHEN and WHY those patterns solve problems that no other technique handles cleanly, and where the monotonic stack and sliding window maximum ideas turn these simple structures into genuinely powerful algorithmic tools.

## Prerequisites
- `02_Recursion` (Phase 01) — the call stack is a stack, literally; this notebook makes that connection explicit and shows you how to replace recursion with an explicit stack
- `03_STL_Deep_Dive` (Phase 00, Sections 5 and 7) — `std::deque`, `std::stack`, `std::queue` as container adapters
- `02_Sliding_Window_and_Prefix_Sum` (Phase 02) — the sliding window technique returns in Section 4, enhanced by a deque
- `07_Linked_Lists` (Phase 02) — the transition from this notebook references stacks/queues as the next step

## Learning Objectives
By the end of this notebook, you will be able to:
- Explain what LIFO and FIFO access patterns mean concretely, and why each is suited to fundamentally different problem shapes
- Implement a stack from scratch and use it for balanced parentheses checking and expression evaluation
- Implement a queue from scratch (both array-based circular and linked-list-based) and explain why the circular buffer avoids wasted space
- Apply the monotonic stack technique to solve "next greater element" and similar problems in O(n)
- Apply a deque to solve the sliding window maximum problem in O(n), and explain why a plain stack or queue cannot
- Convert any recursive DFS into an iterative version using an explicit stack, connecting directly to `02_Recursion`'s stack-overflow discussion
- Use a queue for BFS (level-order traversal) and explain why FIFO ordering is what makes BFS work
- Choose between stack, queue, and deque for a given problem based on the required access pattern

## Checklist Before Moving On
- [ ] I can explain why a stack is the right structure for matching parentheses, in one sentence
- [ ] I can explain why a circular buffer is better than a naive array-based queue
- [ ] I can trace through a monotonic stack on a concrete example and explain what invariant the stack maintains
- [ ] I can implement the sliding window maximum using a deque and explain why it's O(n) amortized
- [ ] I can convert a recursive function to iterative using an explicit stack
- [ ] I can explain why BFS requires a queue (FIFO) and DFS maps to a stack (LIFO)

## Table of Contents
1. Stack Fundamentals — LIFO, the Call Stack Made Explicit
2. Classic Stack Applications — Balanced Parentheses and Expression Evaluation
3. Monotonic Stack — Next Greater Element in O(n)
4. Queue Fundamentals — FIFO and the Circular Buffer
5. BFS With a Queue — Why FIFO Guarantees Level Order
6. Deque — Sliding Window Maximum in O(n)
7. Stack-Based DFS vs Recursive DFS
8. When to Use Stack vs Queue vs Deque
9. Phase Checkpoint, Cheat Sheet, and Answer Key
10. LeetCode Practice Problems

## 1. Stack Fundamentals — LIFO, the Call Stack Made Explicit

### The Why
You've been using a stack since `02_Recursion` (Phase 01) without building one yourself — every recursive call pushes a frame onto the call stack, and every return pops one off. This section makes that implicit structure explicit: a stack is any structure with Last-In-First-Out access, meaning the most recently added element is the first one removed. Understanding this concretely — not just as a definition — is what lets you recognize stack-shaped problems instantly.

### Core Mechanism
- **LIFO (Last In, First Out):** `push` adds to the top, `pop` removes from the top, `top` peeks at the top — all O(1). There is no access to anything except the top element without popping everything above it first.
- **The call stack IS a stack:** when `02_Recursion` described each recursive call as "pushing a frame" and each return as "popping a frame," that was literal, not metaphorical. The OS maintains a stack data structure for every running thread; stack overflow happens when this physical stack runs out of space, which is exactly why `02_Recursion` Section 5 discussed it as a real constraint.
- **Implementation choices:** a stack can be backed by a dynamic array (`vector`), a linked list, or (in C++) the `std::stack` adapter from `03_STL_Deep_Dive` Section 7, which wraps `deque` by default. The choice affects cache performance but not the O(1) guarantee for push/pop/top — the abstract access pattern (LIFO) is what defines it as a stack, not the underlying storage.
- **Why LIFO solves certain problems:** any situation where you need to "match" or "undo" things in reverse order — matching an opening bracket with its NEAREST unmatched closing bracket, evaluating nested expressions from the innermost outward, undoing the most recent action first — is structurally a stack problem, because the thing you need to process next is always the thing that arrived most recently.

### Common Pitfalls
- Calling `.top()` or `.pop()` on an empty stack — unlike `vector`, `std::stack` does NOT throw an exception; it's undefined behavior, and it will crash or silently corrupt memory. Always check `.empty()` first.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <stack>
using namespace std;

// A stack is just LIFO access -- push to the top, pop from the top, peek at the
// top. No access to anything below the top without removing everything above it.
// You've been using one implicitly since 02_Recursion: the call stack IS a stack.

// IMPLEMENTING A STACK FROM SCRATCH: backed by a vector (dynamic array). This is
// essentially what std::stack<int, vector<int>> does internally. The vector handles
// resizing; push/pop/top are all O(1) amortized.
class MyStack {
    vector<int> data;
public:
    void push(int val) { data.push_back(val); }       // O(1) amortized
    void pop()         { data.pop_back(); }            // O(1)
    int top()          { return data.back(); }         // O(1)
    bool empty()       { return data.empty(); }        // O(1)
    int size()         { return data.size(); }         // O(1)
};

int main() {
    // --- FROM-SCRATCH VERSION ---
    MyStack s;
    s.push(10);
    s.push(20);
    s.push(30);

    cout << "from-scratch stack (LIFO order): ";
    while (!s.empty()) {
        cout << s.top() << " ";   // 30, 20, 10 -- LIFO: last pushed = first popped
        s.pop();
    }
    cout << "\n";

    // --- STL VERSION (03_STL_Deep_Dive, Section 7) ---
    stack<int> stl_s;
    stl_s.push(10);
    stl_s.push(20);
    stl_s.push(30);

    cout << "STL stack (LIFO order):          ";
    while (!stl_s.empty()) {
        cout << stl_s.top() << " ";   // same output: 30, 20, 10
        stl_s.pop();
    }
    cout << "\n";

    // THE CONNECTION TO 02_RECURSION: the call stack that powers every recursive
    // function works EXACTLY like this -- each function call pushes a "frame" onto
    // the stack, each return pops one off. Stack overflow (02_Recursion Section 5)
    // happens when too many frames are pushed without popping. Section 7 of THIS
    // notebook shows how to replace the implicit call stack with an explicit
    // std::stack to avoid that problem.

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. Classic Stack Applications — Balanced Parentheses and Expression Evaluation

### The Why
These two problems are the canonical demonstrations of WHY LIFO access is the right fit. Balanced parentheses is the simplest stack problem — every opening bracket waits on the stack for its NEAREST matching closer, and "nearest" means "most recently pushed," which is exactly what `.top()` gives you. Postfix expression evaluation extends the same insight to arithmetic: operators act on the two most recently seen operands, which again means the operands you need are sitting on top of the stack.

### Core Mechanism
- **Balanced parentheses:** push every opening bracket. When you see a closing bracket, pop the top and check it matches — if it doesn't, or if the stack is empty (nothing to match against), the string is unbalanced. At the end, the stack must be empty (every opener was matched).
- **WHY this works:** nesting requires that the INNERMOST pair is matched first, and the innermost opener is always the most recent one — which is the top of the stack. This is why a queue (FIFO) would fail here: it would try to match the OUTERMOST opener first, which is wrong for nested structures.
- **Postfix (Reverse Polish Notation) evaluation:** push operands onto the stack. When you see an operator, pop the top two operands, apply the operator, and push the result back. The final answer is the single value remaining on the stack.
- **WHY postfix avoids parentheses entirely:** the order of operations is encoded in the ORDER of the tokens, not in explicit parentheses or precedence rules. The infix expression `(3 + 4) * 2` becomes `3 4 + 2 *` in postfix — the `+` appears immediately after its operands, before `*` appears after ITS operands, which implicitly encodes that `+` happens first.

### Common Pitfalls
- In balanced parentheses: forgetting to check if the stack is empty BEFORE calling `.top()` when a closing bracket appears — an unmatched closer arrives when there's nothing to match it against, and popping an empty stack is undefined behavior.
- In postfix evaluation: popping operands in the wrong order — the FIRST pop is the RIGHT operand, the SECOND pop is the LEFT operand. For commutative operators (`+`, `*`) this doesn't matter, but for `-` and `/` it produces the wrong result.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <stack>
#include <string>
#include <vector>
#include <sstream>
using namespace std;

// BALANCED PARENTHESES: push every opener, pop and match on every closer.
// WHY a stack: nesting means the INNERMOST pair must match first, and the
// innermost opener is the most recent one -- i.e. the top of the stack.
bool isBalanced(const string& s) {
    stack<char> st;
    for (char c : s) {
        if (c == '(' || c == '[' || c == '{') {
            st.push(c);   // opening bracket: wait for its match
        } else if (c == ')' || c == ']' || c == '}') {
            if (st.empty()) return false;   // closer with no opener to match it
            char top = st.top(); st.pop();
            if ((c == ')' && top != '(') ||
                (c == ']' && top != '[') ||
                (c == '}' && top != '{'))
                return false;   // mismatched types
        }
    }
    return st.empty();   // all openers matched? if not, there are unmatched openers left
}

// POSTFIX (REVERSE POLISH NOTATION) EVALUATION: push operands, pop two on seeing
// an operator, apply it, push the result. No parentheses or precedence rules needed
// because order-of-operations is baked into the token order itself.
int evalPostfix(vector<string>& tokens) {
    stack<int> st;
    for (const string& tok : tokens) {
        if (tok == "+" || tok == "-" || tok == "*" || tok == "/") {
            int right = st.top(); st.pop();   // FIRST pop = RIGHT operand
            int left  = st.top(); st.pop();   // SECOND pop = LEFT operand
            if      (tok == "+") st.push(left + right);
            else if (tok == "-") st.push(left - right);
            else if (tok == "*") st.push(left * right);
            else                 st.push(left / right);   // integer division
        } else {
            st.push(stoi(tok));   // operand: push it and wait
        }
    }
    return st.top();   // the single remaining value is the final answer
}

int main() {
    // --- BALANCED PARENTHESES ---
    cout << "\"({[]})\" balanced? " << isBalanced("({[]})") << "\n";   // 1 (true)
    cout << "\"({[}])\" balanced? " << isBalanced("({[}])") << "\n";   // 0 (false -- mismatched)
    cout << "\"((()\"  balanced? " << isBalanced("((()" ) << "\n";   // 0 (false -- unclosed)

    // --- POSTFIX EVALUATION ---
    // infix: (3 + 4) * 2 = 14
    // postfix: 3 4 + 2 *
    vector<string> expr1{"3", "4", "+", "2", "*"};
    cout << "3 4 + 2 * = " << evalPostfix(expr1) << "\n";   // 14

    // infix: 5 + ((1 + 2) * 4) - 3 = 14
    // postfix: 5 1 2 + 4 * + 3 -
    vector<string> expr2{"5", "1", "2", "+", "4", "*", "+", "3", "-"};
    cout << "5 1 2 + 4 * + 3 - = " << evalPostfix(expr2) << "\n";   // 14

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Monotonic Stack — Next Greater Element in O(n)

### The Why
The brute-force approach to "for each element, find the next element to the right that is greater" is O(n²) — for every element, scan rightward. The monotonic stack reduces this to O(n) by maintaining a stack that is always in decreasing order (top to bottom). This is one of the most important stack patterns in competitive programming and interviews, and it generalizes to any "next greater/smaller" variant.

### Core Mechanism
- **The invariant:** the stack always holds indices of elements in DECREASING order of their values (bottom to top). Every element that enters the stack is guaranteed to be smaller than the element below it — if it isn't, we pop until the invariant is restored.
- **Why this solves "next greater element":** when we encounter a new element `arr[i]` that is GREATER than `arr[st.top()]`, we've found the answer for `st.top()` — the next greater element to its right is `arr[i]`. We pop `st.top()`, record the answer, and keep popping as long as the invariant is violated.
- **Why it's O(n):** each element is pushed onto the stack EXACTLY ONCE and popped EXACTLY ONCE, across the entire algorithm. So even though there's a while-loop inside the for-loop, the total number of push+pop operations across ALL iterations is at most 2n — this is the same amortized argument as `02_Sliding_Window_and_Prefix_Sum`'s shrinking step.
- **Monotonic INCREASING variant:** flip the comparison. A stack that maintains increasing order (bottom to top) solves "next SMALLER element" instead. The same O(n) amortized argument applies.
- **The pattern generalizes:** "previous greater element," "next smaller or equal," histogram problems (largest rectangle in histogram), daily temperatures — all are variations on the same monotonic stack idea, with the comparison direction and the direction of traversal adjusted.

### Common Pitfalls
- Storing VALUES on the stack instead of INDICES — you often need the index to record the answer in the right position of the result array. Store indices; access values via `arr[st.top()]`.
- Forgetting the elements that remain on the stack after the loop completes — they have no "next greater element" (nothing to the right was larger), and their answer should be -1 (or whatever the problem's sentinel is).

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <stack>
using namespace std;

// NEXT GREATER ELEMENT using a MONOTONIC STACK (decreasing, bottom to top).
// For each element, find the first element to its RIGHT that is strictly greater.
// If none exists, the answer is -1.
//
// WHY O(n): each index is pushed once and popped once -- at most 2n operations
// total, same amortized argument as 02_Sliding_Window_and_Prefix_Sum's shrink step.
vector<int> nextGreaterElement(vector<int>& arr) {
    int n = arr.size();
    vector<int> result(n, -1);   // default: no greater element found
    stack<int> st;               // stores INDICES, not values

    for (int i = 0; i < n; i++) {
        // pop everything that arr[i] is greater than -- arr[i] IS their answer
        while (!st.empty() && arr[st.top()] < arr[i]) {
            result[st.top()] = arr[i];   // arr[i] is the next greater element for st.top()
            st.pop();
        }
        st.push(i);   // push current index -- it still needs its own answer
    }
    // anything left on the stack has no next greater element -- result already -1
    return result;
}

// DAILY TEMPERATURES: same idea, but instead of returning the greater VALUE,
// return the NUMBER OF DAYS until a warmer temperature. Same monotonic stack,
// just computing `i - st.top()` instead of `arr[i]`.
vector<int> dailyTemperatures(vector<int>& temps) {
    int n = temps.size();
    vector<int> result(n, 0);   // default: no warmer day ahead
    stack<int> st;

    for (int i = 0; i < n; i++) {
        while (!st.empty() && temps[st.top()] < temps[i]) {
            result[st.top()] = i - st.top();   // how many days until warmer
            st.pop();
        }
        st.push(i);
    }
    return result;
}

int main() {
    vector<int> arr{4, 5, 2, 10, 8};
    vector<int> nge = nextGreaterElement(arr);

    cout << "arr:                ";
    for (int x : arr) cout << x << " ";
    cout << "\n";

    cout << "next greater:       ";
    for (int x : nge) cout << x << " ";
    cout << "\n";   // expected: 5 10 10 -1 -1

    vector<int> temps{73, 74, 75, 71, 69, 72, 76, 73};
    vector<int> dt = dailyTemperatures(temps);

    cout << "temps:              ";
    for (int x : temps) cout << x << " ";
    cout << "\n";

    cout << "days until warmer:  ";
    for (int x : dt) cout << x << " ";
    cout << "\n";   // expected: 1 1 4 2 1 1 0 0

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Queue Fundamentals — FIFO and the Circular Buffer

### The Why
A queue is the structural opposite of a stack: First-In-First-Out instead of Last-In-First-Out. Where a stack processes the MOST RECENT item first (matching brackets, undoing actions), a queue processes items IN THE ORDER THEY ARRIVED — which is exactly what you need when fairness or ordering matters (task scheduling, BFS traversal, message processing). The circular buffer is the standard trick for implementing a queue efficiently on top of an array — without it, a naive array queue wastes space or requires O(n) shifts.

### Core Mechanism
- **FIFO (First In, First Out):** `push` adds to the back, `pop` removes from the front — both O(1). The first element enqueued is the first element dequeued.
- **Naive array problem:** if you use a plain array and always dequeue from index 0, you must shift everything left — O(n) per dequeue. Alternatively, you can track a `front` pointer and avoid shifting, but then the usable portion of the array marches rightward, wasting all the space behind `front`.
- **Circular buffer fix:** when `front` or `back` reaches the end of the array, wrap around to index 0 using modular arithmetic (`(index + 1) % capacity`). This reuses the empty space at the beginning, giving O(1) enqueue and dequeue with no wasted space — the array is treated as a ring, not a line.
- **Linked-list alternative:** a singly linked list with a maintained `head` (front) and `tail` (back) pointer also gives O(1) enqueue (append at tail) and O(1) dequeue (remove from head) — no capacity limit, but worse cache performance than the contiguous circular buffer.
- **C++ STL:** `std::queue` (`03_STL_Deep_Dive`, Section 7) is backed by `std::deque` by default, which handles all of this internally.

### Common Pitfalls
- In a circular buffer: confusing "full" and "empty" — both have `front == back` if you're not careful. The standard fix is to either track a separate `count` variable or to waste one slot (the buffer is full when `(back + 1) % capacity == front`).
- Forgetting that `std::queue::pop()` does NOT return the removed element — you must call `.front()` FIRST to read the value, THEN `.pop()` to remove it. Same design as `std::stack`.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <queue>
using namespace std;

// CIRCULAR BUFFER QUEUE: an array-based queue that wraps around using modular
// arithmetic, avoiding both the O(n)-shift problem and the wasted-space problem
// of a naive array queue.
class CircularQueue {
    int* data;
    int front_idx, back_idx, count, capacity;
public:
    CircularQueue(int cap) : capacity(cap), front_idx(0), back_idx(0), count(0) {
        data = new int[cap];
    }
    ~CircularQueue() { delete[] data; }

    void push(int val) {
        // in production code, you'd resize or reject here if full
        data[back_idx] = val;
        back_idx = (back_idx + 1) % capacity;   // WRAP AROUND using modular arithmetic --
                                                  // this is the entire circular buffer trick
        count++;
    }

    int pop() {
        int val = data[front_idx];
        front_idx = (front_idx + 1) % capacity;   // advance front, wrapping if needed
        count--;
        return val;
    }

    int front() { return data[front_idx]; }
    bool empty() { return count == 0; }
    int size() { return count; }
};

int main() {
    // --- FROM-SCRATCH CIRCULAR QUEUE ---
    CircularQueue q(10);
    q.push(10);
    q.push(20);
    q.push(30);

    cout << "circular queue (FIFO order): ";
    while (!q.empty()) {
        cout << q.pop() << " ";   // 10, 20, 30 -- FIFO: first pushed = first popped
    }
    cout << "\n";

    // demonstrate wrap-around: push 8 items, pop 5, push 5 more
    CircularQueue q2(8);
    for (int i = 1; i <= 8; i++) q2.push(i * 10);
    for (int i = 0; i < 5; i++) q2.pop();   // removes 10,20,30,40,50 -- front wraps
    for (int i = 9; i <= 13; i++) q2.push(i * 10);   // back wraps around to reuse space

    cout << "after wrap-around:           ";
    while (!q2.empty()) {
        cout << q2.pop() << " ";   // 60,70,80,90,100,110,120,130 -- all correct
    }
    cout << "\n";

    // --- STL VERSION (03_STL_Deep_Dive, Section 7) ---
    queue<int> stl_q;
    stl_q.push(10);
    stl_q.push(20);
    stl_q.push(30);

    cout << "STL queue (FIFO order):      ";
    while (!stl_q.empty()) {
        cout << stl_q.front() << " ";   // .front() reads, .pop() removes -- two separate calls
        stl_q.pop();
    }
    cout << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. BFS With a Queue — Why FIFO Guarantees Level Order

### The Why
Breadth-First Search (BFS) is the fundamental graph/tree traversal that explores nodes level by level — all nodes at distance 1 from the start before any node at distance 2, all at distance 2 before any at distance 3, and so on. It's the canonical application of a queue, and the reason FIFO ordering is structurally necessary (not just convenient) is worth understanding now, even though the full BFS treatment comes in `03_Tree_DFS_BFS` (Phase 03) and `Phase_04_Graph_Theory`. This section gives you the mechanism and the intuition.

### Core Mechanism
- **BFS algorithm:** start by enqueueing the root/source. Repeat: dequeue the front node, process it, enqueue all its unvisited neighbors. Stop when the queue is empty.
- **Why FIFO guarantees level-order:** all neighbors of a node at distance `d` from the source are at distance `d+1`. Since FIFO processes nodes in the order they were enqueued, and all distance-`d` nodes are enqueued BEFORE their distance-`(d+1)` children, every node at distance `d` is processed before any node at distance `d+1`. This is exactly the level-by-level guarantee.
- **Why a STACK (LIFO) would break this:** if you replaced the queue with a stack, you'd get DFS instead — diving deep into one branch before exploring siblings at the same level. DFS is a different, equally important traversal (Section 7), but it does NOT guarantee level-order or shortest-path-first processing.
- **Grid BFS:** the exact same algorithm applies to 2D grids (treat each cell as a node, neighbors are adjacent cells) — this is the pattern behind `03_Backtracking`'s grid traversal, but non-recursive and guaranteeing shortest paths.
- **Complexity:** O(V + E) time (every vertex and every edge examined once), O(V) space (the queue can hold up to one full level of the graph). This is optimal for exploring the entire reachable graph.

### Common Pitfalls
- Forgetting to mark nodes as visited WHEN THEY ARE ENQUEUED, not when they are dequeued — if you mark on dequeue, the same node can be enqueued multiple times by different neighbors before it gets processed, wasting time and potentially causing incorrect results.
- Using BFS when DFS would suffice (or vice versa): BFS guarantees shortest paths in unweighted graphs; DFS does not. If shortest path doesn't matter and you just need reachability, either works.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <queue>
using namespace std;

// BFS on a simple graph (adjacency list). This is a PREVIEW of Phase 03/04's
// full treatment -- the key insight here is WHY the queue's FIFO ordering is
// what guarantees level-by-level exploration.

void bfs(vector<vector<int>>& adj, int start) {
    int n = adj.size();
    vector<bool> visited(n, false);
    vector<int> dist(n, -1);   // distance from start, -1 = not yet reached

    queue<int> q;
    q.push(start);
    visited[start] = true;   // mark visited ON ENQUEUE, not on dequeue -- this
                              // prevents the same node from being enqueued multiple
                              // times by different neighbors
    dist[start] = 0;

    while (!q.empty()) {
        int cur = q.front();
        q.pop();

        for (int neighbor : adj[cur]) {
            if (!visited[neighbor]) {
                visited[neighbor] = true;   // mark BEFORE pushing
                dist[neighbor] = dist[cur] + 1;
                q.push(neighbor);
            }
        }
    }

    // print distances
    for (int i = 0; i < n; i++) {
        cout << "node " << i << ": distance = " << dist[i] << "\n";
    }
}

// GRID BFS: find shortest path from top-left to bottom-right in a 0/1 grid
// (0 = passable, 1 = wall). Same BFS algorithm, neighbors are up/down/left/right.
int shortestPathGrid(vector<vector<int>>& grid) {
    int rows = grid.size(), cols = grid[0].size();
    if (grid[0][0] == 1 || grid[rows-1][cols-1] == 1) return -1;

    vector<vector<bool>> visited(rows, vector<bool>(cols, false));
    queue<tuple<int,int,int>> q;   // {row, col, distance}
    q.push({0, 0, 1});
    visited[0][0] = true;

    int dx[] = {0, 0, 1, -1};
    int dy[] = {1, -1, 0, 0};

    while (!q.empty()) {
        auto [r, c, d] = q.front();
        q.pop();

        if (r == rows-1 && c == cols-1) return d;

        for (int i = 0; i < 4; i++) {
            int nr = r + dx[i], nc = c + dy[i];
            if (nr >= 0 && nr < rows && nc >= 0 && nc < cols &&
                !visited[nr][nc] && grid[nr][nc] == 0) {
                visited[nr][nc] = true;
                q.push({nr, nc, d + 1});
            }
        }
    }
    return -1;   // no path exists
}

int main() {
    // --- GRAPH BFS ---
    //   0 -- 1 -- 3
    //   |         |
    //   2 ------- 4
    vector<vector<int>> adj(5);
    adj[0] = {1, 2};
    adj[1] = {0, 3};
    adj[2] = {0, 4};
    adj[3] = {1, 4};
    adj[4] = {2, 3};

    cout << "BFS from node 0:\n";
    bfs(adj, 0);
    // expected: node 0: dist=0, node 1: dist=1, node 2: dist=1, node 3: dist=2, node 4: dist=2

    // --- GRID BFS ---
    vector<vector<int>> grid = {
        {0, 0, 0},
        {1, 1, 0},
        {0, 0, 0}
    };
    cout << "\nshortest path in grid: " << shortestPathGrid(grid) << "\n";   // expected: 5

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Deque — Sliding Window Maximum in O(n)

### The Why
`02_Sliding_Window_and_Prefix_Sum` (Phase 02) introduced the sliding window pattern, where a fixed-or-variable-size window moves across an array. Finding the MAXIMUM value within each window position is a natural extension — the brute-force approach scans the entire window per step (O(nk) total), but a deque (double-ended queue, `03_STL_Deep_Dive` Section 5) reduces this to O(n) total by maintaining a clever invariant.

### Core Mechanism
- **The problem:** given an array and a window size `k`, report the maximum value in every contiguous window of size `k` as the window slides from left to right.
- **Deque as a "monotonic decreasing window":** maintain a deque of INDICES. The front of the deque is always the index of the current window's maximum. The invariant: values at indices in the deque are in DECREASING order (front to back).
- **Processing each new element `arr[i]`:**
  1. **Remove from the back** while `arr[deque.back()] <= arr[i]` — these elements can NEVER be a window maximum because `arr[i]` is larger and entered later (so it will outlast them in the window). This maintains the decreasing invariant.
  2. **Push `i` to the back.**
  3. **Remove from the front** if `deque.front()` is outside the current window (`deque.front() <= i - k`).
  4. **The answer** for the current window is `arr[deque.front()]` — the front always holds the maximum.
- **Why it's O(n):** each index is pushed to the deque at most once and popped at most once (from either end) — at most 2n total operations, same amortized reasoning as the monotonic stack (Section 3) and `02_Sliding_Window_and_Prefix_Sum`'s window shrinking.
- **Why a plain stack or queue can't do this:** a stack can only remove from ONE end; a queue can only add to one end and remove from the other. The deque needs to remove from BOTH ends (back: to maintain the decreasing invariant; front: to expire elements that have left the window) — this dual-end access is the specific capability that makes the O(n) solution possible.

### Common Pitfalls
- Using `<` instead of `<=` when popping from the back — using strict `<` leaves duplicates of the maximum in the deque, which is harmless for correctness but wastes space. Using `<=` is cleaner: if the new element equals the current back, the back can't be a future maximum anyway (it will leave the window first or at the same time).
- Forgetting to check whether the deque's front has fallen outside the current window before reading the answer — this produces stale maximums from previous windows.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <deque>
using namespace std;

// SLIDING WINDOW MAXIMUM using a MONOTONIC DEQUE.
// For each window of size k, report the maximum value.
//
// The deque stores INDICES. Values at those indices are in DECREASING order
// (front = largest). Each index is pushed once and popped once -- O(n) total.
vector<int> maxSlidingWindow(vector<int>& arr, int k) {
    vector<int> result;
    deque<int> dq;   // indices, with arr[dq.front()] >= arr[dq.back()] always

    for (int i = 0; i < (int)arr.size(); i++) {
        // STEP 1: remove from the back while the new element is >= the back's value.
        // These elements can NEVER be a future maximum because arr[i] is at least as
        // large AND entered later (will outlast them in the window).
        while (!dq.empty() && arr[dq.back()] <= arr[i]) {
            dq.pop_back();
        }

        // STEP 2: push current index
        dq.push_back(i);

        // STEP 3: remove from the front if it's outside the current window
        if (dq.front() <= i - k) {
            dq.pop_front();
        }

        // STEP 4: once we've filled the first window (i >= k-1), the front is the max
        if (i >= k - 1) {
            result.push_back(arr[dq.front()]);
        }
    }
    return result;
}

// SLIDING WINDOW MINIMUM: exact same technique, just flip the comparison
// (maintain INCREASING order instead of decreasing).
vector<int> minSlidingWindow(vector<int>& arr, int k) {
    vector<int> result;
    deque<int> dq;

    for (int i = 0; i < (int)arr.size(); i++) {
        while (!dq.empty() && arr[dq.back()] >= arr[i]) {   // flip: >= instead of <=
            dq.pop_back();
        }
        dq.push_back(i);
        if (dq.front() <= i - k) dq.pop_front();
        if (i >= k - 1) result.push_back(arr[dq.front()]);
    }
    return result;
}

int main() {
    vector<int> arr{1, 3, -1, -3, 5, 3, 6, 7};
    int k = 3;

    cout << "array: ";
    for (int x : arr) cout << x << " ";
    cout << "\n";

    vector<int> maxes = maxSlidingWindow(arr, k);
    cout << "window max (k=" << k << "): ";
    for (int x : maxes) cout << x << " ";
    cout << "\n";   // expected: 3 3 5 5 6 7

    vector<int> mins = minSlidingWindow(arr, k);
    cout << "window min (k=" << k << "): ";
    for (int x : mins) cout << x << " ";
    cout << "\n";   // expected: -1 -3 -3 -3 3 3

    // WHY THIS IS O(n): each index is pushed to the deque at most once and popped
    // at most once (from either end). Total operations across all iterations: at
    // most 2n pushes + 2n pops. Same amortized argument as 02_Sliding_Window's
    // window shrinking and Section 3's monotonic stack.

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. Stack-Based DFS vs Recursive DFS

### The Why
`02_Recursion` (Phase 01, Section 5) explained that every recursive call implicitly uses the OS call stack, and deep recursion risks stack overflow. The fix — converting to an EXPLICIT stack — was mentioned there as a technique; this section shows the concrete implementation. The structural insight: DFS IS a stack algorithm, whether that stack is the implicit call stack or an explicit `std::stack`. Once you see this, the duality between DFS/stack and BFS/queue becomes completely clear.

### Core Mechanism
- **Recursive DFS** uses the call stack implicitly — each recursive call pushes a frame, each return pops one. The LIFO ordering of function returns is what causes depth-first behavior: you go as deep as possible before backtracking.
- **Iterative DFS** replaces the call stack with an explicit `std::stack`. Push the start node; repeat: pop the top, process it, push its unvisited neighbors. The result is the same depth-first traversal order (with minor edge-case differences in neighbor ordering).
- **The key equivalence:** replacing the queue in BFS with a stack gives you DFS. Replacing the stack in DFS with a queue gives you BFS. The DATA STRUCTURE is the only difference — the loop structure is identical.
- **When to prefer iterative DFS:** whenever the recursion depth could be large enough to overflow the call stack — graphs with millions of nodes, or deeply nested tree structures. The explicit stack lives on the heap, which is vastly larger than the OS stack.
- **The space trade-off:** recursive DFS uses O(depth) call-stack space (each frame holds local variables + return address — typically 100+ bytes per frame on most platforms). Iterative DFS uses O(depth) heap space (each entry on the explicit stack is typically just a single integer or small struct — much more compact).

### Common Pitfalls
- Marking nodes as visited on POP instead of on PUSH for iterative DFS (same issue as BFS Section 5's pitfall) — this allows the same node to be pushed onto the stack multiple times by different neighbors. For correctness it still works (you just skip already-visited nodes when you pop them), but it wastes space and time.

In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <stack>
using namespace std;

// RECURSIVE DFS: uses the call stack implicitly. Each recursive call = push,
// each return = pop. The LIFO ordering of returns is what makes it depth-first.
void dfsRecursive(vector<vector<int>>& adj, vector<bool>& visited, int node) {
    visited[node] = true;
    cout << node << " ";
    for (int neighbor : adj[node]) {
        if (!visited[neighbor]) {
            dfsRecursive(adj, visited, neighbor);
        }
    }
}

// ITERATIVE DFS: replaces the implicit call stack with an explicit std::stack.
// The loop structure is IDENTICAL to BFS (Section 5), with `stack` instead of `queue`.
// That single change -- LIFO instead of FIFO -- is the ENTIRE structural difference
// between DFS and BFS.
void dfsIterative(vector<vector<int>>& adj, int start) {
    int n = adj.size();
    vector<bool> visited(n, false);
    stack<int> st;

    st.push(start);
    // NOTE: for iterative DFS, marking on pop is common (and correct, though it
    // can push duplicates). Marking on push also works and avoids duplicates.
    // Both approaches give a valid DFS traversal.

    while (!st.empty()) {
        int node = st.top();
        st.pop();

        if (visited[node]) continue;   // skip if already processed (handles duplicates)
        visited[node] = true;
        cout << node << " ";

        // push neighbors in REVERSE order so that the leftmost neighbor is processed
        // first (to match recursive DFS order, which visits neighbors in forward order)
        for (int i = adj[node].size() - 1; i >= 0; i--) {
            if (!visited[adj[node][i]]) {
                st.push(adj[node][i]);
            }
        }
    }
}

int main() {
    //   0 -- 1 -- 3
    //   |         |
    //   2 ------- 4
    vector<vector<int>> adj(5);
    adj[0] = {1, 2};
    adj[1] = {0, 3};
    adj[2] = {0, 4};
    adj[3] = {1, 4};
    adj[4] = {2, 3};

    cout << "recursive DFS from 0: ";
    vector<bool> visited(5, false);
    dfsRecursive(adj, visited, 0);
    cout << "\n";

    cout << "iterative DFS from 0: ";
    dfsIterative(adj, 0);
    cout << "\n";

    // THE KEY INSIGHT: the ONLY structural difference between the BFS code in
    // Section 5 and this iterative DFS is replacing `queue` with `stack`. FIFO vs
    // LIFO is the entire difference between breadth-first and depth-first traversal.

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 8. When to Use Stack vs Queue vs Deque

### The Why
Stacks, queues, and deques are all "restricted-access" data structures — they deliberately limit HOW you access data compared to a full-access array. The restrictions aren't limitations; they're the feature. Choosing the right one means recognizing which access pattern your problem's logic structurally requires.

### Core Mechanism
- **Use a STACK (LIFO) when:** the problem requires processing the MOST RECENT unprocessed item first. Key signals: matching/nesting (parentheses, tags), undo/backtrack, DFS, any "innermost first" structure, expression evaluation, monotonic stack patterns ("next greater/smaller").
- **Use a QUEUE (FIFO) when:** the problem requires processing items IN THE ORDER THEY ARRIVED. Key signals: BFS, level-order traversal, task scheduling, any "first come first served" structure, anything where you need to guarantee that closer/earlier items are handled before farther/later ones.
- **Use a DEQUE when:** you need efficient access at BOTH ends. Key signals: sliding window maximum/minimum (Section 6), any situation where elements can be removed from either end based on different conditions (back: monotonic invariant maintenance; front: window expiration). `std::deque` also serves as the default backing container for both `std::stack` and `std::queue` in C++ — it's the most general of the three.
- **None of these when:** you need random access to arbitrary positions (use `vector`), or you need frequent insertions/deletions at arbitrary positions given a pointer (use a linked list, `07_Linked_Lists`).

### Common Pitfalls
- Reaching for a deque when a stack or queue would suffice — `deque` has slightly more overhead than a plain `vector`-backed stack. Use the simplest structure that matches your problem's access pattern.
- Confusing "deque" (the data structure, double-ended queue) with "dq" (common variable name for deque). In C++, `std::deque` is a full container with random access; `std::queue` and `std::stack` are adapters that restrict its interface.

## 9. Phase Checkpoint, Cheat Sheet, and Answer Key

### Stack / Queue / Deque Cheat Sheet

| Structure | Access Pattern | Push | Pop | Peek | Typical Use Case |
|---|---|---|---|---|---|
| Stack | LIFO | O(1) to top | O(1) from top | O(1) top only | Parentheses, DFS, undo, monotonic stack |
| Queue | FIFO | O(1) to back | O(1) from front | O(1) front only | BFS, scheduling, level-order traversal |
| Deque | Both ends | O(1) to either end | O(1) from either end | O(1) either end | Sliding window max/min, general double-ended needs |

### Monotonic Stack / Deque Pattern Summary

| Pattern | Structure | Invariant | Solves |
|---|---|---|---|
| Monotonic decreasing stack | Stack | Top is always smallest | Next greater element, daily temperatures |
| Monotonic increasing stack | Stack | Top is always largest | Next smaller element |
| Monotonic decreasing deque | Deque | Front is always largest | Sliding window maximum |
| Monotonic increasing deque | Deque | Front is always smallest | Sliding window minimum |

### BFS vs DFS Structural Comparison

| Property | BFS (Queue) | DFS (Stack/Recursion) |
|---|---|---|
| Data structure | Queue (FIFO) | Stack (LIFO) / call stack |
| Traversal order | Level by level | Branch by branch (deepest first) |
| Shortest path (unweighted) | Yes, guaranteed | No |
| Space | O(width of graph) | O(depth of graph) |
| Stack overflow risk | No (heap-allocated queue) | Yes, if recursive and depth is large |

### Checkpoint Questions
1. Why is a stack (not a queue) the right structure for checking balanced parentheses?
2. What invariant does a monotonic decreasing stack maintain, and why does that invariant solve "next greater element"?
3. Why does a circular buffer solve the wasted-space problem of a naive array-based queue?
4. Why does FIFO ordering guarantee that BFS processes nodes level by level?
5. What specific property of a deque (that neither a stack nor a queue has) enables the O(n) sliding window maximum algorithm?
6. What is the ONLY structural change needed to convert the BFS loop into a DFS loop?

### Answer Key
1. Matching brackets is a nesting problem — the INNERMOST pair must be matched first, and the innermost opener is always the most RECENTLY pushed one, which is the stack's top. A queue would try to match the outermost (oldest) opener first, which is wrong for nested structures.
2. The stack maintains the invariant that values are in decreasing order from bottom to top. When a new element `arr[i]` violates this invariant (it's greater than the top), the top's "next greater element" has been found — it's `arr[i]`. This resolves all pending elements smaller than `arr[i]` in one pass.
3. A naive array queue either wastes space (front marches rightward, leaving dead space behind) or requires O(n) shifts to repack. A circular buffer reuses the dead space by wrapping indices around using `(index + 1) % capacity`, treating the array as a ring.
4. All neighbors of distance-d nodes are at distance d+1. FIFO ensures all distance-d nodes are dequeued (and their neighbors enqueued) before any distance-(d+1) node is dequeued — so every node at distance d is fully processed before any node at distance d+1, which IS level-order.
5. The deque supports removal from BOTH ends: pop_back (to maintain the monotonic decreasing invariant when a larger element arrives) AND pop_front (to expire elements that have left the sliding window). Neither a stack (one-end access) nor a queue (add-one-end, remove-other-end, but not arbitrary removal from the add-end) can do both.
6. Replace `queue` with `stack`. The loop body is identical — pop from the structure, process the node, push its neighbors. FIFO (queue) gives breadth-first; LIFO (stack) gives depth-first. That's the entire difference.

### Mini Project
Implement a MinStack: a stack that supports `push`, `pop`, `top`, and `getMin` — all in O(1) time.
1. Maintain TWO stacks: the main stack (holds all values) and an auxiliary stack (holds the current minimum at each level of the main stack).
2. On `push(val)`: push `val` onto the main stack; push `min(val, aux.top())` onto the auxiliary stack.
3. On `pop()`: pop both stacks.
4. `getMin()`: return `aux.top()` — always O(1) since the auxiliary stack's top always reflects the current minimum.

This exercises the core stack idea (LIFO for tracking state) combined with a clever auxiliary structure — a pattern that recurs in more complex interview problems.

## 10. LeetCode Practice Problems

Grouped by pattern. Hints point at the pattern's key decision, not the full solution.

### Basic Stack Operations

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 20 | Valid Parentheses | Easy | Section 2, directly — push openers, pop and match on closers |
| 155 | Min Stack | Medium | Section 9's Mini Project — maintain an auxiliary stack tracking the running minimum |
| 150 | Evaluate Reverse Polish Notation | Medium | Section 2's postfix evaluation, directly |
| 71 | Simplify Path | Medium | Stack-based — push directory names, pop on `..`, ignore `.` and empty segments |

### Monotonic Stack

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 739 | Daily Temperatures | Medium | Section 3, directly — monotonic decreasing stack, recording distance instead of value |
| 496 | Next Greater Element I | Easy | Section 3's next greater element, with a hash map to connect the two arrays |
| 503 | Next Greater Element II | Medium | Circular array variant — process the array twice (or use `i % n`) to handle wrap-around |
| 84 | Largest Rectangle in Histogram | Hard | Monotonic increasing stack — for each bar, find the nearest shorter bar to its left and right |
| 42 | Trapping Rain Water | Hard | Can be solved with monotonic stack (track bars in decreasing order) or two pointers (`01_Arrays_and_Two_Pointers`) |

### Queue and BFS

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 933 | Number of Recent Calls | Easy | Queue-based — keep a queue of timestamps, pop from front when outside the 3000ms window |
| 200 | Number of Islands | Medium | Grid BFS (Section 5) — start BFS from each unvisited `1`, mark the entire island as visited |
| 994 | Rotting Oranges | Medium | Multi-source BFS — enqueue ALL initially rotten oranges, BFS outward simultaneously |
| 752 | Open the Lock | Medium | BFS on a graph of lock states — each state has 8 neighbors (4 digits × 2 directions) |

### Sliding Window with Deque

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 239 | Sliding Window Maximum | Hard | Section 6, directly — monotonic decreasing deque |
| 862 | Shortest Subarray with Sum at Least K | Hard | Monotonic deque on prefix sums — deque maintains increasing prefix sums; pop front when current prefix minus front's prefix >= K |

### Stack-Based DFS

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 94 | Binary Tree Inorder Traversal | Easy | Iterative with explicit stack — push all left children first, process on pop, then move to right child |
| 144 | Binary Tree Preorder Traversal | Easy | Section 7's iterative DFS pattern — push right child first so left is processed first |
| 145 | Binary Tree Postorder Traversal | Easy | Trickiest iterative variant — use two stacks or a modified single-stack approach with a "last visited" pointer |

### Self-Check Before Moving to Phase 03
If you can implement a monotonic stack from scratch, explain why BFS requires a queue and DFS maps to a stack, and solve the sliding window maximum with a deque — you've internalized the three core ideas of this notebook. Phase 03 (`01_Heaps_and_Priority_Queues`, `02_Trees_and_BST`, `03_Tree_DFS_BFS`) will build directly on these: heaps generalize the "always access the extreme element" idea, and tree traversals are the concrete setting where BFS (queue) and DFS (stack/recursion) see their heaviest use.